# 03_research_preflight

This notebook diagnoses multi-task conflict *before* a long training run. It measures dataset coverage, one-batch gradient conflict, and branch feature divergence.

In [ ]:
!pip install -q torch torchvision torchmetrics pyyaml scipy opencv-python matplotlib tqdm


In [ ]:
import os, sys, torch
from google.colab import drive
drive.mount('/content/drive')
ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
PROJECT_DIR = os.path.join(ECOCAR_ROOT, 'DETR_GeoLane_pipeline')
sys.path.insert(0, PROJECT_DIR)
print('GPU available:', torch.cuda.is_available())


In [ ]:
from src.config import Config, find_lane_labels
from src.dataset import build_dataloaders
from src.runtime_paths import ensure_local_dataset_from_drive

DATASET_NAME = 'bdd100k_vehicle5'
LOCAL_DS = ensure_local_dataset_from_drive(
    dataset_name=DATASET_NAME,
    ecocar_root=ECOCAR_ROOT,
)
print(f"Notebook-local dataset root: {LOCAL_DS}")

train_lane_json = find_lane_labels('train')
val_lane_json = find_lane_labels('val')
print('Train lane labels:', train_lane_json)
print('Val lane labels:', val_lane_json)

cfg = Config(
    run_name='dualpath_v3_research_stage',
    dataset_root=LOCAL_DS,
    img_size=640,
    batch_size=4,
    num_workers=4,
    lane_num_queries=8,
    lane_points=64,
    use_task_adapters=True,
    cross_attn=True,
    cross_attn_start_epoch=8,
    det_only_epochs=3,
    auto_resume=False,
)
train_loader, val_loader = build_dataloaders(cfg, train_lane_json=train_lane_json, val_lane_json=val_lane_json)
print(len(train_loader.dataset), len(val_loader.dataset))

In [ ]:
from src.model import build_model
from src.losses import DualPathLoss
from src.research_tools import gradient_conflict_summary, feature_branch_gap

model = build_model(cfg).to(cfg.device)
criterion = DualPathLoss(cfg)
batch = next(iter(train_loader))
images = batch['images'].to(cfg.device)
batch_gpu = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
model.set_epoch(0)
outputs = model(images)
conflict = gradient_conflict_summary(model, criterion, outputs, batch_gpu)
feature_gap = feature_branch_gap(model, images[:2])
print('gradient conflict:', conflict)
print('feature gap:', feature_gap)


In [ ]:
lane_ratio = float(batch['has_lanes'].float().mean().item())
print('lane_supervision_ratio_in_batch =', lane_ratio)
